# RUN ON COLAB (free T4 GPU) — TensorRT FP16 / INT8 conversion + benchmark

Before running: **Runtime > Change runtime type > T4 GPU**.

This is the one part of the optimization phase that genuinely needs a GPU
— TensorRT builds an engine specific to the GPU it's built on, so it can't
be built or benchmarked on the CPU-only path the rest of `optimization/`
uses. This notebook writes `results/tensorrt_benchmark.json` in the same
schema `optimization/benchmark.py` uses for every other backend, so you can
copy it back into the repo and merge it with:
```
python -m optimization.benchmark ... --extra-results results/tensorrt_benchmark.json
```

**Honesty note**: this notebook was written against the documented TensorRT
8/10 Python API but has not been executed — there is no GPU in the
environment this project was developed in. Expect to debug specific API
calls (TensorRT version differences are common) on first real run.

In [ ]:
!git clone https://github.com/DharshanaReddy/sanding-region-segmentation.git /content/repo
%cd /content/repo
!pip install -q -e ".[train]"
!pip install -q tensorrt pycuda onnx

In [ ]:
# Bring your own trained checkpoint + exported ONNX model (from export_onnx.py,
# run locally or in a previous Colab session), and your generated dataset.
ONNX_PATH = '/content/drive/MyDrive/sanding-region-segmentation/deeplabv3_mobilenet.onnx'
DATA_DIR = '/content/data'  # unzip your dataset here first, same as colab_train.ipynb
IMAGE_SIZE = 512
NUM_CALIB_SAMPLES = 100

In [ ]:
import time
import platform

import numpy as np
import pycuda.autoinit  # noqa: F401 - registers the CUDA context
import pycuda.driver as cuda
import tensorrt as trt
import torch

from training.dataset import NUM_CLASSES, SegmentationDataset
from training.metrics import ConfusionMatrixTracker

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

In [ ]:
class SegmentationEntropyCalibrator(trt.IInt8EntropyCalibrator2):
    """Calibrates on the same normalized training images as the ONNX Runtime
    static quantizer and OpenVINO's NNCF path, so all three INT8 rows in the
    benchmark table are calibrated on equivalent data."""

    def __init__(self, data_dir, image_size, num_samples, batch_size=1):
        super().__init__()
        dataset = SegmentationDataset(data_dir, 'train', image_size=image_size, augment=False)
        n = min(num_samples, len(dataset))
        self.batches = [dataset[i][0].unsqueeze(0).numpy().astype(np.float32) for i in range(n)]
        self.batch_size = batch_size
        self.index = 0
        self.device_input = cuda.mem_alloc(self.batches[0].nbytes) if self.batches else None

    def get_batch_size(self):
        return self.batch_size

    def get_batch(self, names):
        if self.index >= len(self.batches):
            return None
        cuda.memcpy_htod(self.device_input, np.ascontiguousarray(self.batches[self.index]))
        self.index += 1
        return [int(self.device_input)]

    def read_calibration_cache(self):
        return None

    def write_calibration_cache(self, cache):
        pass

In [ ]:
def build_engine(onnx_path, precision, data_dir=None, image_size=512, num_calib=100):
    """precision: 'fp16' or 'int8'."""
    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, TRT_LOGGER)
    with open(onnx_path, 'rb') as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError('Failed to parse ONNX model')

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1GB

    if precision == 'fp16':
        config.set_flag(trt.BuilderFlag.FP16)
    elif precision == 'int8':
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)  # allow FP16 fallback for layers INT8 can't handle well
        config.int8_calibrator = SegmentationEntropyCalibrator(data_dir, image_size, num_calib)
    else:
        raise ValueError(precision)

    engine_bytes = builder.build_serialized_network(network, config)
    runtime = trt.Runtime(TRT_LOGGER)
    return runtime.deserialize_cuda_engine(engine_bytes)

In [ ]:
def benchmark_engine(engine, image_size, warmup=10, iterations=50):
    context = engine.create_execution_context()
    input_shape = (1, 3, image_size, image_size)
    output_shape = context.get_tensor_shape(engine.get_tensor_name(1))

    h_input = np.random.randn(*input_shape).astype(np.float32)
    h_output = np.empty(output_shape, dtype=np.float32)
    d_input = cuda.mem_alloc(h_input.nbytes)
    d_output = cuda.mem_alloc(h_output.nbytes)
    stream = cuda.Stream()

    context.set_tensor_address(engine.get_tensor_name(0), int(d_input))
    context.set_tensor_address(engine.get_tensor_name(1), int(d_output))

    def infer(x):
        cuda.memcpy_htod_async(d_input, np.ascontiguousarray(x), stream)
        context.execute_async_v3(stream_handle=stream.handle)
        cuda.memcpy_dtoh_async(h_output, d_output, stream)
        stream.synchronize()
        return h_output.copy()

    for _ in range(warmup):
        infer(h_input)

    latencies_ms = []
    for _ in range(iterations):
        start = time.perf_counter()
        infer(h_input)
        latencies_ms.append((time.perf_counter() - start) * 1000)

    arr = np.array(latencies_ms)
    mean_ms = float(arr.mean())
    return {
        'mean_latency_ms': mean_ms,
        'median_latency_ms': float(np.median(arr)),
        'p95_latency_ms': float(np.percentile(arr, 95)),
        'fps': 1000.0 / mean_ms,
        'peak_memory_mb': float(cuda.mem_get_info()[1] - cuda.mem_get_info()[0]) / 1e6,
    }, infer

In [ ]:
def measure_miou(infer_fn, data_dir, image_size, max_samples=None):
    dataset = SegmentationDataset(data_dir, 'test', image_size=image_size, augment=False)
    tracker = ConfusionMatrixTracker(NUM_CLASSES)
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    for i in range(n):
        image, mask = dataset[i]
        logits = infer_fn(image.unsqueeze(0).numpy())
        pred = torch.from_numpy(logits).argmax(dim=1)
        tracker.update(pred, mask.unsqueeze(0))
    return tracker.mean_iou()

In [ ]:
import json

hardware = f"{torch.cuda.get_device_name(0)} (Colab T4), {platform.processor()}"
results = []
for precision, label in [('fp16', 'TensorRT FP16'), ('int8', 'TensorRT INT8')]:
    print(f'Building {label} engine...')
    engine = build_engine(ONNX_PATH, precision, DATA_DIR, IMAGE_SIZE, NUM_CALIB_SAMPLES)
    stats, infer_fn = benchmark_engine(engine, IMAGE_SIZE)
    miou = measure_miou(infer_fn, DATA_DIR, IMAGE_SIZE)
    results.append({'name': label, 'hardware': hardware, 'miou': miou, **stats})
    print(label, stats, 'mIoU', miou)

with open('/content/tensorrt_benchmark.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Wrote /content/tensorrt_benchmark.json — copy this back into results/ in your local repo.')